# Task assignment


Consider an assignment problem where the goal is to assign n workers to n tasks such that the total cost is minimized. The problem can be formulated as:

$$ \min_x \sum_{i=1}^{n} \sum_{j=1}^{n} c_{ij} x_{ij} $$ $$ \sum_{j=1}^{n} x_{ij} = 1, \qquad \forall i \in \{ 1, \dots , n \} \\ \sum_{i=1}^{n} x_{ij} = 1, \qquad \forall j \in \{ 1, \dots , n \} \\ x_{ij} \in \{ 0, 1 \}, \qquad \forall i,j \in \{ 1, \dots , n \} $$

where $c_{ij}$ is the cost of assigning worker $i$ to task $j$.
Furthermore, $x_{ij} = 1$ if worker $i$ is assigned to task $j$, and $0$ otherwise.

# Distributionally robust optimization

#### 1. Formulate a Wasserstein distributionally robust optimization (DRO) problem with l1 or l∞ norm for the risk-averse objective function.

**Distributional Assumptions**

We assume that the $c_{ij}$ are independent random variables and satisfy:

$$
c_{ij} \sim \mathcal{N}(\mu_{ij}, \sigma_{ij}^2) = \mathbb{P}_{ij}, \quad \forall i,j.
$$

The parameters are defined as:
$$
\mu_{ij} = (n+1)i + j, \qquad
\sigma_{ij} = 1 + (n+1)i + j.
$$

**Formulation of Wasserstein distributionally robust optimization**

We flatten matrix $C_{n \times n}$ and $X_{n \times n}$ into vectors in then following way:
$$
\sum_{i=1}^{n} \sum_{j=1}^{n} c_{ij} x_{ij} = \mathbf{c}^{\top} \mathbf{x}
\\
\mathbf{c} = [c_1, c_2, \ldots, c_n]^\top \in \mathbb{R}^{n^2}
\\
\mathbf{x} = [x_1, x_2, \ldots, x_n]^\top \in \{0, 1\}^{n^2}
$$

Then, we can use general formulation of Risk-Averse Distributionally Robust Optimization(DRO):
$$
\min_{\mathbf{x} \in X, t \in \mathbb{R}} \max_{\mathbb{P} \in \mathcal{P}} \left\{ t + \frac{1}{1 - \alpha} \mathbb{E}_{\mathbb{P}} \left\{ \max \{\mathbf{c}^{\top} \mathbf{x} - t, 0 \} \right\} \right\}
$$

where

$$
U = \{\mathbf{c} \in \mathbb{R}^{n^2} : \mathbf{D}\mathbf{c} \leq \mathbf{g}\}
\\
\hat{C}_{train} = \{\hat{\mathbf{c}}^{(k)},\ k \in \{1, \dots, K\}\} \\
\mathcal{P} := \left\{\mathbb{P} : \mathbb{P}\{\mathbf{c} \in U\} = 1 \ \text{and} \ d_W(\hat{\mathbb{P}}_0, \mathbb{P}) \leq \varepsilon \right\}
\\
\hat{\mathbb{P}}_0(\hat{C}_{train}) := \frac{1}{K} \sum_{k=1}^K \delta_{\hat{\mathbf{c}}^{(k)}}
$$

It can be simplified to:
$$
\begin{align}
\min_{\mathbf{x}, \boldsymbol{\nu}, \mathbf{s}, \lambda, t} & \left\{ t + \frac{1}{1 - \alpha} \left( \lambda \varepsilon + \frac{1}{K} \sum_{k=1}^K s_k \right) \right\}
\\
\text{s.t.} \quad & \mathbf{x} \in \{0, 1\}^{n^2}
\\
& \lambda \ge 0
\\
& \hat{\mathbf{c}}^{(k)\top} \mathbf{x} - t + \boldsymbol{\Delta}^{(k)\top} \boldsymbol{\nu}^{(k)} \le s_k, \quad \forall k \in \{1, \dots, K\}, \nonumber
\\
& \| \mathbf{D}^\top \boldsymbol{\nu}^{(k)} - \mathbf{x} \|_{∞} \le \lambda, \quad \forall k \in \{1, \dots, K\}, \Longleftrightarrow -\lambda \leq \left( \mathbf{D}^\top \boldsymbol{\nu}^{(k)} - \mathbf{x} \right)_i \leq \lambda, \quad \forall i = \overline{1, n^2} \quad \forall k \in \{1, \dots, K\}, \nonumber
\\
& \boldsymbol{\nu}^{(k)} \ge 0, \ s_k \ge 0, \quad \forall k \in \{1, \dots, K\}
\end{align}
$$
where $\boldsymbol{\Delta}^{(k)} = \mathbf{g} - \mathbf{D}\hat{\mathbf{c}}^{(k)} \ge 0$ for each $k \in \{1, \dots, K\}$


In [14]:
import json
import gurobipy as gp
import matplotlib.pyplot as plt
from pathlib import Path
from tqdm import tqdm

GUROBI_CONF_PATH = "./gurobi_conf.json"


def set_environ(environ: gp.Env, conf_path: str = GUROBI_CONF_PATH):
    with open(conf_path, "r") as cfp:
        configuration = json.load(cfp)

    access_id = configuration["environ"].get("WLSAccessID")
    secret = configuration["environ"].get("WLSSecret")
    licence_id = configuration["environ"].get("LicenseID")

    print(f"NOTE: Got environment configuration parameters:\nWLSAccessID: {access_id},\nWLSSecret: {secret},\nLicenseID: {licence_id}\n")
    if all([access_id, secret, licence_id]):
        print(f"NOTE: found existing Gurobi licence")
        environ.setParam('WLSAccessID', access_id)
        environ.setParam('WLSSecret', secret)
        environ.setParam('LicenseID', licence_id)
    else:
        print(f"WARNING: some of the environment configuration parameters are None - using free Gurobi licence: you will not be able to solve large problems!")


def set_model_params(model: gp.Model, conf_path: str = GUROBI_CONF_PATH):
    with open(conf_path, "r") as cfp:
        configuration = json.load(cfp)

    output_flag = configuration["model"].get("OutputFlag", 0)
    log_to_console = configuration["model"].get("LogToConsole", 0)

    model.setParam("OutputFlag", output_flag)
    model.setParam("LogToConsole", log_to_console)

In [3]:
import numpy as np
from gurobipy import GRB


def build_box_uncertainty(samples):
    # bounds chosen by samples min/max
    l = samples.min(axis=0)
    u = samples.max(axis=0)
    dim = len(l)
    D = np.vstack([np.eye(dim), -np.eye(dim)])
    g = np.hstack([u, -l])
    return D, g


def assignment_constraints(model, x, n):
    def idx(i, j):
        return i * n + j

    for i in range(n):
        model.addConstr(sum(x[idx(i, j)] for j in range(n)) == 1)

    for j in range(n):
        model.addConstr(sum(x[idx(i, j)] for i in range(n)) == 1)


class DROAssignment:
    def __init__(self, n, epsilon, alpha, D, g, env):
        self.n = n
        self.epsilon = epsilon
        self.alpha = alpha
        self.D = D
        self.g = g
        self.env = env

    def solve(self, samples):
        K, dim = samples.shape
        n = self.n

        model = gp.Model("DRO_Assignment", env=self.env)
        set_model_params(model)

        # variables
        x = model.addVars(dim, vtype=GRB.BINARY, name="x")
        t = model.addVar(lb=-GRB.INFINITY, name="t")
        lam = model.addVar(lb=0, name="lambda")
        s = model.addVars(K, lb=0, name="s")

        nu = {}
        for k in range(K):
            nu[k] = model.addVars(self.D.shape[0], lb=0, name=f"nu_{k}")

        # assignment constraints
        assignment_constraints(model, x, n)

        # DRO constraints
        for k in range(K):
            c_k = samples[k]

            # Delta_k = g - D c_k
            Delta_k = self.g - self.D @ c_k

            # main constraint
            model.addConstr(
                gp.quicksum(c_k[i] * x[i] for i in range(dim))
                - t
                + gp.quicksum(Delta_k[j] * nu[k][j] for j in range(len(Delta_k)))
                <= s[k]
            )

            # infinity norm constraints
            for i in range(dim):
                expr = (
                    gp.quicksum(self.D[j, i] * nu[k][j] for j in range(self.D.shape[0]))
                    - x[i]
                )
                model.addConstr(expr <= lam)
                model.addConstr(expr >= -lam)

        # objective
        model.setObjective(
            t + (1 / (1 - self.alpha)) * (lam * self.epsilon + (1 / K) * gp.quicksum(s[k] for k in range(K))),
            GRB.MINIMIZE
        )

        model.optimize()

        x_sol = np.array([x[i].X for i in range(dim)])
        t_val = t.X
        return x_sol.reshape((n, n)), model.objVal, t_val

#### 2. For each n ∈ {5, 10, . . . , 50}, generate 100 test instances of the problem by using k = 30 samples from the true distribution.

In [4]:
class DRODataGenerator:
    def __init__(self, seed=42):
        """
        Initialize random number generator for reproducibility.
        """
        self.rng = np.random.default_rng(seed)

    def generate_samples(self, n, K):
        """
        Generate K samples from the true distribution P*.

        Each sample corresponds to a cost matrix C (n x n),
        which is flattened into a vector of length n^2.

        Distribution:
            c_ij ~ N(mu_ij, sigma_ij^2)

        where:
            mu_ij = (n+1)*i + j
            sigma_ij = 1 + (n+1)*i + j

        Parameters:
            n (int): number of workers/tasks
            K (int): number of samples

        Returns:
            samples (np.ndarray): shape (K, n^2)
        """
        M = n + 1
        samples = []

        for _ in range(K):
            C = np.zeros((n, n))

            for i in range(n):
                for j in range(n):
                    # Note: using (i+1), (j+1) for 1-based indexing
                    mu = M * (i + 1) + (j + 1)
                    sigma = 1 + M * (i + 1) + (j + 1)

                    # Draw from normal distribution
                    C[i, j] = self.rng.normal(mu, sigma)

            # Flatten matrix into vector
            samples.append(C.flatten())

        return np.array(samples)  # shape: (K, n^2)

    def generate_experiment(self, n_values, num_instances=100, K=30):
        """
        Generate full experimental dataset.

        For each n in n_values:
            - create num_instances independent problem instances
            - each instance consists of K samples

        Parameters:
            n_values (list): list of problem sizes (e.g., [5, 10, ..., 50])
            num_instances (int): number of instances per n
            K (int): number of samples per instance

        Returns:
            data (dict):
                data[n] = list of instances
                each instance has shape (K, n^2)
        """
        data = {}

        for n in n_values:
            instances = []

            for _ in range(num_instances):
                samples = self.generate_samples(n, K)
                instances.append(samples)

            data[n] = instances

        return data

In [5]:
# simple usage example
env = gp.Env(empty=True)  # Initialize an empty environment
# Set license parameters using the environment
set_environ(env)
# Now, initialize the environment
env.start()

generator = DRODataGenerator(seed=1)
n_values = range(5, 51, 5)
results = {}
for n in n_values:
    results[n] = []
    for instance_id in range(100):
        # 1. generate data
        samples = generator.generate_samples(n, K=30)

        # 2. build uncertainty set
        D, g = build_box_uncertainty(samples)

        # 3. solve DRO
        solver = DROAssignment(
            n=n,
            epsilon=0.1,
            alpha=0.95,
            D=D,
            g=g,
            env=env
        )

        x_hat, obj, t_hat = solver.solve(samples)

        results[n].append({
            "x": x_hat,
            "obj": obj,
            "t": t_hat,
        })
        print(f"{x_hat=}, {obj=}, {t_hat=}")
env.close()

NOTE: Got environment configuration parameters:
WLSAccessID: c883e76c-6b5d-4a16-9dc8-dd44f341af22,
WLSSecret: 1721a792-2ef0-4ae2-a7c7-993621b5fcca,
LicenseID: 2799306

NOTE: found existing Gurobi licence
Set parameter WLSAccessID
Set parameter WLSSecret
Set parameter LicenseID to value 2799306
Academic license 2799306 - for non-commercial use only - registered to se___@business.uzh.ch
x_hat=array([[ 1., -0.,  0., -0., -0.],
       [-0.,  1.,  0., -0., -0.],
       [-0.,  0., -0.,  0.,  1.],
       [-0.,  0.,  0.,  1.,  0.],
       [-0.,  0.,  1., -0., -0.]]), obj=131.50482746611735, t_hat=123.73118400739148
x_hat=array([[0., 0., 1., 0., 0.],
       [1., 0., 0., 0., 0.],
       [0., 0., 0., 0., 1.],
       [0., 1., 0., 0., 0.],
       [0., 0., 0., 1., 0.]]), obj=160.59306934961626, t_hat=155.05057808744465
x_hat=array([[-0.,  0.,  0.,  0.,  1.],
       [-0., -0.,  0.,  1., -0.],
       [-0.,  1., -0., -0., -0.],
       [ 0., -0.,  1., -0., -0.],
       [ 1., -0., -0., -0., -0.]]), obj=1

KeyboardInterrupt: 

#### 3. Explore the dependence of the average out-of-sample performance (with standard deviations) on the Wasserstein radius, ε. Find the critical Wasserstein radius ε∗ that minimizes the theoretical out-of-sample performance.

In [ ]:
# create "theoretical" plot of dependence out-of-sample performance on Wasserstein radius, there must be an estimation of ε critical point here

In [6]:
def solve_saa_assignment(samples, n, env):
    K, dim = samples.shape

    model = gp.Model(env=env)
    set_model_params(model)

    x = model.addVars(dim, vtype=GRB.BINARY)

    assignment_constraints(model, x, n)

    mean_c = samples.mean(axis=0)

    model.setObjective(
        gp.quicksum(mean_c[i] * x[i] for i in range(dim)),
        GRB.MINIMIZE
    )

    model.optimize()

    x_sol = np.array([x[i].X for i in range(dim)])
    return x_sol

In [7]:
def cvar_loss(x, t, samples, alpha):
    x = np.array(x).reshape(-1)   # FIX
    losses = []
    for c in samples:
        c = np.array(c).reshape(-1)  # FIX
        z = np.dot(c, x)
        losses.append(t + (1/(1-alpha)) * max(z - t, 0))

    return np.mean(losses)


def compute_z_star(samples, n, env):
    x_star = solve_saa_assignment(samples, n, env)

    costs = [np.dot(c, x_star.flatten()) for c in samples]
    return np.mean(costs)

In [21]:
def append_result_to_json(path: Path, n, eps, mean_rho, std_rho):
    path = Path(path)
    path.parent.mkdir(exist_ok=True, parents=True)

    if path.exists():
        with open(path, "r") as f:
            data = json.load(f)
    else:
        data = {}

    n_key = str(n)

    if n_key not in data:
        data[n_key] = []

    data[n_key].append({
        "epsilon": float(eps),
        "mean_rho": float(mean_rho),
        "std_rho": float(std_rho)
    })

    with open(path, "w") as f:
        json.dump(data, f, indent=4)

In [23]:
def run_eps_estimation(env, generator, n_values, eps_grid, num_instances=20, K_train=30, K_test=1000, json_path="task3_results.json"):
    results = {}
    for n in n_values:
        print(f"\n=== n = {n} ===")
        eps_results = []
        for eps in eps_grid:
            print(f"  epsilon = {eps}")
            rho_instances = []
            for instance_id in tqdm(range(num_instances)):

                train = generator.generate_samples(n, K_train)
                test = generator.generate_samples(n, K_test)

                D, g = build_box_uncertainty(train)

                solver = DROAssignment(
                    n=n,
                    epsilon=eps,
                    alpha=0.95,
                    D=D,
                    g=g,
                    env=env
                )

                x_hat, obj, t_hat = solver.solve(train)
                out_sample = cvar_loss(x_hat, t_hat, test, alpha=0.95)

                z_star = compute_z_star(test, n, env=env)

                rho = out_sample / z_star
                rho_instances.append(rho)

            mean_rho = np.mean(rho_instances)
            std_rho = np.std(rho_instances)

            eps_results.append((mean_rho, std_rho, eps))
            append_result_to_json(
                json_path,
                n,
                eps,
                mean_rho,
                std_rho
            )

        results[n] = eps_results

    return results

In [24]:
def plot_results(results, eps_grid, n_values):
    plt.figure(figsize=(10, 6))

    for n in n_values:
        means = [r[0] for r in results[n]]
        stds = [r[1] for r in results[n]]

        plt.errorbar(eps_grid, means, yerr=stds, label=f"n={n}")

    plt.xlabel("Wasserstein radius ε")
    plt.ylabel("ρ_out")
    plt.title("Out-of-sample performance vs ε")
    plt.legend()
    plt.grid()
    plt.show()

In [26]:
env = gp.Env(empty=True)
set_environ(env)
env.start()

generator = DRODataGenerator(seed=1)

NOTE: Got environment configuration parameters:
WLSAccessID: c883e76c-6b5d-4a16-9dc8-dd44f341af22,
WLSSecret: 1721a792-2ef0-4ae2-a7c7-993621b5fcca,
LicenseID: 2799306

NOTE: found existing Gurobi licence
Set parameter WLSAccessID
Set parameter WLSSecret
Set parameter LicenseID to value 2799306
Academic license 2799306 - for non-commercial use only - registered to se___@business.uzh.ch


In [ ]:
n_values = [5]
eps_grid = np.linspace(0.0, 10.0, 10)

results = run_eps_estimation(
    env=env,
    generator=generator,
    n_values=n_values,
    eps_grid=eps_grid,
    num_instances=50,
    K_train=30,
    K_test=1000,
    json_path="./optimal_eps_estimation/eps_for_n5.json",
)
plot_results(results, eps_grid, n_values)

In [ ]:
n_values = [10]
eps_grid = np.linspace(30.0, 100.0, 10)

results = run_eps_estimation(
    env=env,
    generator=generator,
    n_values=n_values,
    eps_grid=eps_grid,
    num_instances=10,
    K_train=15,
    K_test=200,
    json_path="./optimal_eps_estimation/eps_for_n10_30-100.json",
)
plot_results(results, eps_grid, n_values)


=== n = 10 ===
  epsilon = 30.0


100%|██████████| 10/10 [00:52<00:00,  5.24s/it]


  epsilon = 37.77777777777778


100%|██████████| 10/10 [02:09<00:00, 12.95s/it]


  epsilon = 45.55555555555556


...

100%|██████████| 10/10 [01:16<00:00,  7.69s/it]


  epsilon = 68.88888888888889


100%|██████████| 10/10 [00:56<00:00,  5.63s/it]


  epsilon = 76.66666666666666


 70%|███████   | 7/10 [00:43<00:17,  5.94s/it]

In [ ]:
env.close()

#### 4. Find the average critical Wasserstein radius using cross validation (holdout method and/or k-fold cross validation [3]) and compare this radius with the theoretical estimate ε∗.

In [ ]:
# find "practical" ε critical point by k-fold cross validation and compare with "theoretical" estimation(3.)

#### 5. Explore how the critical Wasserstein radius ε∗ depends on the sample size, k.

In [ ]:
# create plot of dependence of "theoretical" estimation of ε critical point(it is ~1/sqrt(k)) on the sample size, k

#### 6. For fixed n, compute the average solution times as a function of ε.

In [ ]:
# plots with dependence of time spent on the solution on the ε

#### 7. For fixed ε, compute the average solution times as a function of n. Compare the obtained solution times with the SAA (ε = 0) and the robust optimization approach (ε is sufficiently large).

In [ ]:
# plots with dependence of time spent on the solution on the n and plot with comparing solution times with (ε = 0 and ε is sufficiently large)